# 04 · Lắp ráp và kiểm tra PBIP

Repo hiện có report/model mẫu nhưng chưa có writer TMDL/PBIR. Notebook này không giả vờ chuyển layout JSON thành report: nó nhận file contents do bước build tạo ra, chép nền từ mẫu, thay toàn bộ model tables/pages cũ, kiểm tra artifact, rồi mới ghi dự án `.pbip`. Mở bằng Power BI Desktop và Refresh vẫn là bước xác nhận cuối.

In [1]:
from pathlib import Path
import json
import re
import shutil

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'README.md').exists() and (candidate / 'src').is_dir():
            return candidate
    raise FileNotFoundError('Không tìm thấy repo root.')

ROOT = find_repo_root()
RUNS_DIR = ROOT / 'runs'
JOB_ID = None  # Có thể đặt job_id chính xác để build một lần chạy cũ.
required = ('requirement.json', 'data_catalog.json', 'model_spec.json', 'layout.json')
candidates = [p for p in RUNS_DIR.glob('*') if all((p / name).exists() for name in required)]
if JOB_ID:
    RUN_DIR = RUNS_DIR / JOB_ID
elif candidates:
    RUN_DIR = max(candidates, key=lambda p: p.stat().st_mtime)
else:
    raise FileNotFoundError('Thiếu file đầu vào. Chạy notebook 01, 02, 03 và chọn layout trước.')

JOB_ID = RUN_DIR.name

requirement = json.loads((RUN_DIR / 'requirement.json').read_text(encoding='utf-8'))
catalog = json.loads((RUN_DIR / 'data_catalog.json').read_text(encoding='utf-8'))
model = json.loads((RUN_DIR / 'model_spec.json').read_text(encoding='utf-8'))
layout = json.loads((RUN_DIR / 'layout.json').read_text(encoding='utf-8'))
samples = ROOT / 'data' / 'data_samples'
report_templates = list(samples.glob('*.Report'))
model_templates = list(samples.glob('*.SemanticModel'))
if len(report_templates) != 1 or len(model_templates) != 1:
    raise FileNotFoundError('Cần đúng một thư mục mẫu *.Report và *.SemanticModel trong data/data_samples/.')
REPORT_TEMPLATE, MODEL_TEMPLATE = report_templates[0], model_templates[0]
print('Job:', JOB_ID)
print('Report template:', REPORT_TEMPLATE)
print('Model template:', MODEL_TEMPLATE)

FileNotFoundError: Thiếu file đầu vào. Chạy notebook 01, 02, 03 và chọn layout trước.

## Build artifact request
Gửi prompt dưới đây cho coding AI để tạo file contents theo model/layout đã duyệt. Yêu cầu AI không sửa field, DAX hay nghiệp vụ ngoài spec. Phản hồi cần là JSON có `project_files`: object ánh xạ đường dẫn tương đối trong thư mục Report/SemanticModel sang toàn bộ nội dung file. Không đưa file thực thi tùy ý.

In [ ]:
report_json = REPORT_TEMPLATE / 'definition' / 'report.json'
sample_visuals = list((REPORT_TEMPLATE / 'definition' / 'pages').glob('*/visuals/*/visual.json'))
sample_visual = json.loads(sample_visuals[0].read_text(encoding='utf-8')) if sample_visuals else {}
artifact_prompt = f'''Tạo file artifacts cho Power BI Project (PBIP) dùng định dạng PBIR/TMDL hiện đại, dựa trên requirement, data catalog, model spec và layout đã duyệt.
Chỉ trả về JSON: {{"project_files": {{"<Report-folder>/definition/report.json": "toàn bộ text file", "<Report-folder>/definition/pages/pages.json": "...", "<Report-folder>/definition/pages/<page-id>/page.json": "...", "<Report-folder>/definition/pages/<page-id>/visuals/<visual-id>/visual.json": "...", "<SemanticModel-folder>/definition/model.tmdl": "...", "<SemanticModel-folder>/definition/tables/<table>.tmdl": "..."}}}}.
Tên folder là <PROJECT>.Report và <PROJECT>.SemanticModel; tên project: {requirement.get('requirement', {}).get('report_name', 'PBI_Report')}. Bao gồm pages.json, mỗi page.json/visual.json, model.tmdl và đầy đủ table TMDL. PBIR phải dùng field/measure đúng spec, query tham chiếu đúng tên table. Semantic model phải đọc file nguồn {catalog.get('source_file')!r}; không giữ đường dẫn nguồn của template. Không tạo DAX/measure ngoài spec.
Dùng cấu trúc mẫu dưới đây làm tham khảo schema, không sao chép dữ liệu nghiệp vụ mẫu. Giữ JSON tương thích PBIR.

MẪU report.json:
{report_json.read_text(encoding='utf-8') if report_json.exists() else '{}'}

MẪU visual.json:
{json.dumps(sample_visual, ensure_ascii=False, indent=2)}

REQUIREMENT:
{json.dumps(requirement, ensure_ascii=False, indent=2)}
DATA CATALOG:
{json.dumps(catalog, ensure_ascii=False, indent=2)}
MODEL SPEC:
{json.dumps(model, ensure_ascii=False, indent=2)}
LAYOUT ĐƯỢC CHỌN:
{json.dumps(layout, ensure_ascii=False, indent=2)}
'''
print(artifact_prompt)

In [ ]:
artifact_prompt = f'''Tạo file artifacts cho Power BI Project (PBIP) dùng định dạng PBIR/TMDL hiện đại, dựa trên requirement, data catalog, model spec và layout đã duyệt.
Chỉ trả về JSON: {{"project_files": {{"{report_folder_name}/definition/report.json": "toàn bộ text file", "{report_folder_name}/definition/pages/pages.json": "...", "{report_folder_name}/definition/pages/<page-id>/page.json": "...", "{report_folder_name}/definition/pages/<page-id>/visuals/<visual-id>/visual.json": "...", "{model_folder_name}/definition/model.tmdl": "...", "{model_folder_name}/definition/tables/<table>.tmdl": "..."}}}}.
Dùng chính xác tên folder {report_folder_name} và {model_folder_name}. Bao gồm pages.json, mỗi page.json/visual.json, model.tmdl và đầy đủ table TMDL. PBIR phải dùng field/measure đúng spec, query tham chiếu đúng tên table. Semantic model phải đọc file nguồn {catalog.get('source_file')!r}; không giữ đường dẫn nguồn của template. Không tạo DAX/measure ngoài spec.\nDùng cấu trúc mẫu dưới đây làm tham khảo schema, không sao chép dữ liệu nghiệp vụ mẫu. Giữ JSON tương thích PBIR.

MẪU report.json:
{report_json.read_text(encoding='utf-8') if report_json.exists() else '{}'}

MẪU visual.json:
{json.dumps(sample_visual, ensure_ascii=False, indent=2)}

REQUIREMENT:
{json.dumps(requirement, ensure_ascii=False, indent=2)}
DATA CATALOG:
{json.dumps(catalog, ensure_ascii=False, indent=2)}
MODEL SPEC:
{json.dumps(model, ensure_ascii=False, indent=2)}
LAYOUT ĐƯỢC CHỌN:
{json.dumps(layout, ensure_ascii=False, indent=2)}
'''
print(artifact_prompt)

In [ ]:
BUILD_RESPONSE = r'''PASTE_BUILD_JSON_HERE'''

def project_folder_name(value):
    cleaned = re.sub(r'[^A-Za-z0-9 _-]+', '', str(value)).strip().replace(' ', '_')
    return cleaned or 'PBI_Report'

project_name = project_folder_name(requirement.get('requirement', {}).get('report_name', 'PBI_Report'))
report_folder_name = f'{project_name}.Report'
model_folder_name = f'{project_name}.SemanticModel'

def validate_and_build(response_text):
    result = json.loads(response_text)
    files = result.get('project_files')
    if not isinstance(files, dict) or not files:
        raise ValueError('Phản hồi cần có object project_files.')
    allowed_roots = {report_folder_name, model_folder_name}
    normalized_files = {}
    for relative_name, content in files.items():
        relative = Path(relative_name)
        if relative.is_absolute() or '..' in relative.parts or not relative.parts or relative.parts[0] not in allowed_roots:
            raise ValueError(f'Đường dẫn artifact không an toàn/không hợp lệ: {relative_name}')
        if not isinstance(content, str):
            raise ValueError(f'Nội dung file phải là chuỗi: {relative_name}')
        normalized_files[relative.as_posix()] = content
    model_prefix = f'{model_folder_name}/'
    required_report = {
        f'{report_folder_name}/definition/report.json',
        f'{report_folder_name}/definition/pages/pages.json',
    }
    if not required_report.issubset(normalized_files):
        raise ValueError(f'Thiếu file report bắt buộc: {sorted(required_report - normalized_files.keys())}')
    if f'{model_folder_name}/definition/model.tmdl' not in normalized_files:
        raise ValueError('Thiếu SemanticModel/definition/model.tmdl.')
    table_files = [p for p in normalized_files if p.startswith(model_prefix + 'definition/tables/') and p.endswith('.tmdl')]
    page_files = [p for p in normalized_files if re.search(r'/definition/pages/[^/]+/page.json$', p)]
    visual_files = [p for p in normalized_files if re.search(r'/definition/pages/[^/]+/visuals/[^/]+/visual.json$', p)]
    if not table_files or not page_files or not visual_files:
        raise ValueError('Cần ít nhất một table TMDL, page.json và visual.json.')
    for relative_name, content in normalized_files.items():
        if relative_name.endswith('.json'):
            json.loads(content)
    output_root = RUN_DIR / 'output'
    if output_root.exists():
        raise FileExistsError(f'Output đã tồn tại: {output_root}. Đổi JOB_ID hoặc lưu output cũ trước khi build lại.')
    report_root = output_root / report_folder_name
    model_root = output_root / model_folder_name
    output_root.mkdir(parents=True)
    shutil.copytree(REPORT_TEMPLATE, report_root, ignore=shutil.ignore_patterns('.pbi'))
    shutil.copytree(MODEL_TEMPLATE, model_root, ignore=shutil.ignore_patterns('.pbi'))
    shutil.rmtree(report_root / 'definition' / 'pages', ignore_errors=True)
    shutil.rmtree(model_root / 'definition' / 'tables', ignore_errors=True)
    for relative_name, content in normalized_files.items():
        destination = output_root / Path(relative_name)
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_text(content, encoding='utf-8')
    pbir_path = report_root / 'definition.pbir'
    pbir = json.loads(pbir_path.read_text(encoding='utf-8'))
    pbir.setdefault('datasetReference', {}).setdefault('byPath', {})['path'] = f'../{model_folder_name}'
    pbir_path.write_text(json.dumps(pbir, ensure_ascii=False, indent=2), encoding='utf-8')
    descriptor = {'version': '1.0', 'artifacts': [{'report': {'path': report_folder_name}}, {'semanticModel': {'path': model_folder_name}}], 'settings': {'enableAutoRecovery': True}}
    pbip_path = output_root / f'{project_name}.pbip'
    pbip_path.write_text(json.dumps(descriptor, ensure_ascii=False, indent=2), encoding='utf-8')
    for path in report_root.rglob('*.json'):
        json.loads(path.read_text(encoding='utf-8'))
    return pbip_path

if BUILD_RESPONSE.strip() == 'PASTE_BUILD_JSON_HERE':
    print('Dán JSON file artifacts vào BUILD_RESPONSE để lắp ráp. Chưa có file nào được ghi.')
else:
    pbip_path = validate_and_build(BUILD_RESPONSE)
    print('Đã tạo project:', pbip_path)

## Kiểm tra cuối
Notebook chỉ kiểm tra được cấu trúc file và JSON. Hãy mở file `.pbip` bằng Power BI Desktop, kiểm tra quan hệ/DAX, Refresh dữ liệu và xác nhận visual thực sự render; nếu fail, quay lại notebook thiết kế tương ứng.